## Stage 1 — Load input datasets

Load users, purchases, and products CSV files into separate Spark DataFrames and inspect their schemas and sample rows.

### Local Spark Environment Setup

The following environment configuration is specific to the local macOS/Homebrew setup and may need to be adjusted on another machine.

In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"
os.environ["PYSPARK_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("GoIT_DE_HW_03") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 13:28:43 WARN Utils: Your hostname, MacBook-Pro-Valentyn.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.240 instead (on interface en0)
26/10/11 13:28:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 13:28:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
DATA_DIR = "../data"

users_df = spark.read.csv(
    f'{DATA_DIR}/users.csv',
    header=True,
    inferSchema=True
)

purchases_df = spark.read.csv(
    f'{DATA_DIR}/purchases.csv',
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    f'{DATA_DIR}/products.csv',
    header=True,
    inferSchema=True
)

In [4]:
# Inspect schemas
print("Users schema:")
users_df.printSchema()

print("Purchases schema:")
purchases_df.printSchema()

print("Products schema:")
products_df.printSchema()

Users schema:
root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- email: string (nullable = true)

Purchases schema:
root
 |-- purchase_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- quantity: integer (nullable = true)

Products schema:
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [5]:
# Preview data
print("Users:")
users_df.show(5)

print("Purchases:")
purchases_df.show(5)

print("Products:")
products_df.show(5)

Users:
+-------+------+---+-----------------+
|user_id|  name|age|            email|
+-------+------+---+-----------------+
|      1|User_1| 45|user1@example.com|
|      2|User_2| 48|user2@example.com|
|      3|User_3| 36|user3@example.com|
|      4|User_4| 46|user4@example.com|
|      5|User_5| 29|user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
Purchases:
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
+-----------+-------+----------+----------+--------+
only showing top 5 rows
Products:
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+-------

## Stage 2 — Clean Missing Values

Remove rows containing missing values from each input DataFrame.

This ensures that later joins, calculations, and aggregations are performed only on complete records.

In [6]:
# Remove rows with missing values
users_clean_df = users_df.dropna()
purchases_clean_df = purchases_df.dropna()
products_clean_df = products_df.dropna()

# Compare row counts before and after cleaning
print("Users:")
print("Before:", users_df.count())
print("After:", users_clean_df.count())

print("Purchases:")
print("Before:", purchases_df.count())
print("After:", purchases_clean_df.count())

print("Products:")
print("Before:", products_df.count())
print("After:", products_clean_df.count())

Users:
Before: 100
After: 95
Purchases:
Before: 200
After: 195
Products:
Before: 50
After: 47


In [7]:
users_clean_df.show(5)
purchases_clean_df.show(5)
products_clean_df.show(5)

+-------+------+---+-----------------+
|user_id|  name|age|            email|
+-------+------+---+-----------------+
|      1|User_1| 45|user1@example.com|
|      2|User_2| 48|user2@example.com|
|      3|User_3| 36|user3@example.com|
|      4|User_4| 46|user4@example.com|
|      5|User_5| 29|user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
+-----------+-------+----------+----------+--------+
only showing top 5 rows
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------

## Stage 3 — Total Purchase Amount by Product Category

Calculate the total purchase amount for each product category.

Steps:
- join purchases with products using `product_id`;
- calculate the purchase amount as `quantity * price`;
- group the data by product category;
- calculate the total purchase amount for each category.

In [8]:
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

# Join purchases with product information
purchases_products_df = purchases_clean_df.join(
    products_clean_df,
    on="product_id",
    how="inner"
)

# Calculate the total amount for each purchase
purchases_products_df = purchases_products_df.withColumn(
    "purchase_amount",
    col("quantity") * col("price")
)

# Calculate total purchase amount by product category
category_totals_df = purchases_products_df \
    .groupBy("category") \
    .agg(
        spark_round(
            spark_sum("purchase_amount"),
            2
        ).alias("total_purchase_amount")
    )

category_totals_df.show()

+-----------+---------------------+
|   category|total_purchase_amount|
+-----------+---------------------+
|       Home|               1523.5|
|     Sports|               1802.5|
|Electronics|               1174.8|
|   Clothing|                790.3|
|     Beauty|                459.9|
+-----------+---------------------+



## Stage 4 — Purchase Amount by Category for Users Aged 18–25

Calculate the total purchase amount for each product category for users aged from 18 to 25 inclusive.

Steps:
- join purchase and product data with user information using `user_id`;
- filter users aged from 18 to 25;
- group the filtered data by product category;
- calculate the total purchase amount for each category.

In [9]:
# Join purchase data with user information

purchases_users_df = purchases_products_df.join(
    users_clean_df,
    on="user_id",
    how="inner"
)

# Filter users aged from 18 to 25 inclusive

young_users_purchases_df = purchases_users_df.filter(
    col("age").between(18, 25)
)

# Calculate total purchase amount by category

young_category_totals_df = young_users_purchases_df \
    .groupBy("category") \
    .agg(
        spark_round(
            spark_sum("purchase_amount"),
            2
        ).alias("total_purchase_amount_18_25")
    )

young_category_totals_df.show()

+-----------+---------------------------+
|   category|total_purchase_amount_18_25|
+-----------+---------------------------+
|       Home|                      361.1|
|     Sports|                      310.5|
|Electronics|                      249.6|
|   Clothing|                      245.0|
|     Beauty|                       41.4|
+-----------+---------------------------+



## Stage 5 — Spending Share by Category for Users Aged 18–25

Compare total spending for each product category with spending by users aged 18–25.

Calculate each category's share of the total spending made by users aged 18–25.

The percentage is calculated as:

category spending by users aged 18–25
/
total spending by users aged 18–25
× 100

In [10]:
# Calculate total spending for users aged 18–25

total_spending_18_25 = young_category_totals_df \
    .agg(spark_round(
        spark_sum("total_purchase_amount_18_25"),2)
         .alias("total_spending")
    ) \
    .first()["total_spending"]

print("Total spending for users aged 18–25:", total_spending_18_25)

# Combine overall category totals with totals for users aged 18–25

category_percentage_df = category_totals_df.join(
    young_category_totals_df,
    on="category",
    how="inner"
)


# Calculate each category's share of total spending by users aged 18–25

category_percentage_df = category_percentage_df.withColumn(
    "spending_percentage_18_25",
    spark_round(
        col("total_purchase_amount_18_25")
        / total_spending_18_25
        * 100,
        2
    )
)

category_percentage_df.show()

Total spending for users aged 18–25: 1207.6
+-----------+---------------------+---------------------------+-------------------------+
|   category|total_purchase_amount|total_purchase_amount_18_25|spending_percentage_18_25|
+-----------+---------------------+---------------------------+-------------------------+
|       Home|               1523.5|                      361.1|                     29.9|
|     Sports|               1802.5|                      310.5|                    25.71|
|Electronics|               1174.8|                      249.6|                    20.67|
|   Clothing|                790.3|                      245.0|                    20.29|
|     Beauty|                459.9|                       41.4|                     3.43|
+-----------+---------------------+---------------------------+-------------------------+



## Stage 6 — Top 3 Product Categories by Spending Share

Select the three product categories with the highest percentage of spending by users aged from 18 to 25.

Sort the categories by spending percentage in descending order and keep the top three results.

In [11]:
# Select the top 3 categories by spending percentage

top_3_categories_df = category_percentage_df \
    .orderBy(
        col("spending_percentage_18_25").desc()
    ) \
    .limit(3)

top_3_categories_df.show()

+-----------+---------------------+---------------------------+-------------------------+
|   category|total_purchase_amount|total_purchase_amount_18_25|spending_percentage_18_25|
+-----------+---------------------+---------------------------+-------------------------+
|       Home|               1523.5|                      361.1|                     29.9|
|     Sports|               1802.5|                      310.5|                    25.71|
|Electronics|               1174.8|                      249.6|                    20.67|
+-----------+---------------------+---------------------------+-------------------------+

